In [1]:

import sys
import torch
import subprocess

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

subprocess.run(["nvidia-smi"], check=True)

assert torch.cuda.is_available(), "CUDA chưa hoạt động"

print("Số GPU:", torch.cuda.device_count())

device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

# Kiểm tra tính toán và gradient thực tế trên GPU
x = torch.tensor(
    [1.0, 2.0, 3.0],
    device=device,
    requires_grad=True
)

y = (x ** 2).sum()
y.backward()

assert torch.allclose(
    x.grad,
    torch.tensor([2.0, 4.0, 6.0], device=device)
)

torch.cuda.synchronize()
print("GPU_SMOKE_PASS_KAGGLE_ONLY")


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
Thu Nov  5 15:43:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                     

In [2]:

import torch

device = torch.device("cuda:0")

# 1. Kiểm tra convolution CPU/GPU
x = torch.arange(25, dtype=torch.float32).reshape(1, 1, 5, 5)
kernel = torch.tensor([[[[1., 0., -1.],
                         [2., 0., -2.],
                         [1., 0., -1.]]]])

cpu_result = torch.nn.functional.conv2d(x, kernel)
gpu_result = torch.nn.functional.conv2d(
    x.to(device), kernel.to(device)
)

torch.testing.assert_close(gpu_result.cpu(), cpu_result)

print("GPU CONVOLUTION: PASS")

# 2. Kiểm tra huấn luyện mô hình trên GPU
features = torch.tensor(
    [[-2.], [-1.], [0.], [1.], [2.]],
    device=device
)
targets = 2 * features + 1

model = torch.nn.Linear(1, 1, device=device)

with torch.no_grad():
    model.weight.zero_()
    model.bias.zero_()

optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(80):
    optimizer.zero_grad()
    loss = ((model(features) - targets) ** 2).mean()
    loss.backward()
    optimizer.step()

torch.cuda.synchronize()

prediction = model(torch.tensor([[3.]], device=device)).item()

assert abs(prediction - 7.0) < 0.1

print("GPU TRAINING: PASS")
print("Prediction:", prediction)
print("KAGGLE_CUDA_DEEP_SMOKE_PASS")


GPU CONVOLUTION: PASS
GPU TRAINING: PASS
Prediction: 6.999780654907227
KAGGLE_CUDA_DEEP_SMOKE_PASS
